In [1]:
import torch

In [5]:
torch.zeros(4, 8, 16).dim()

3

In [95]:
import math

import torch
from einops import einsum


class Linear(torch.nn.Module):
    weights: torch.Tensor

    def __init__(self, in_features: int, out_features: int, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        self.weights = torch.nn.parameter.Parameter(torch.empty(out_features, in_features, device=device, dtype=dtype))
        std = math.sqrt(2 / (in_features + out_features))
        torch.nn.init.trunc_normal_(tensor=self.weights, mean=0, std=std, a=-3 * std, b=3 * std)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return einsum(x, self.weights, "... in, out in -> ... out")

In [96]:
linear = Linear(1024, 1024)
linear.forward(torch.randn(1024, 1024))

list(linear.state_dict().keys())

['weights']

In [97]:
import torch
import math
from einops import einsum

class Embedding(torch.nn.Module):

    embeddings: torch.Tensor # (num_embeddings, embedding_dim)

    def __init__(self, num_embeddings: int, embedding_dim: int, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        self.embeddings = torch.nn.parameter.Parameter(torch.empty(num_embeddings, embedding_dim, device=device, dtype=dtype))
        torch.nn.init.trunc_normal_(tensor=self.embeddings, mean=0, std=1, a=-3, b=3)

    # token_ids: torch.LongTensor (batch_size, sequence_length)
    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        return self.embeddings[token_ids]

In [215]:
embedding = Embedding(256, 1024)
embedding.forward(torch.randint(0, 256, (2, 5)))

tensor([[[ 0.8381,  0.8891, -0.0872,  ...,  0.2135, -0.3811,  0.7232],
         [-0.8093, -0.3908, -2.0957,  ..., -0.4913,  0.9339, -1.0552],
         [-0.2861, -1.1901,  1.3595,  ...,  1.1698, -0.3780,  0.2114],
         [-0.3864, -0.7551,  0.1825,  ..., -0.1760, -0.5341,  0.3010],
         [-1.1259, -2.0904,  0.4462,  ..., -1.5041, -1.2212,  0.2223]],

        [[-0.5134, -0.3926, -1.4302,  ..., -0.0958,  1.0208, -1.3823],
         [-0.9079,  0.0566, -0.6586,  ..., -0.9603,  0.7266, -0.3156],
         [ 0.2393, -0.3857, -1.7988,  ..., -0.7463, -0.6400,  2.2383],
         [-1.7129, -0.9610,  0.9660,  ..., -0.9184,  0.8813, -0.6655],
         [ 2.3231,  0.1706, -1.0829,  ..., -0.0692, -0.3023, -0.2369]]],
       grad_fn=<IndexBackward0>)

In [99]:
import torch

class RMSNorm(torch.nn.Module):

    gain: torch.Tensor # (d_model, )
    eps: float
    
    def __init__(self, d_model: int, eps: float = 1e-5, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        self.gain = torch.nn.parameter.Parameter(torch.ones(d_model, device=device, dtype=dtype))
        self.eps = eps

    # x (batch_size, sequence_length, d_model)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        in_dtype = x.dtype
        x = x.to(torch.float32)
        x_sqrd_mean = x.pow(2).mean(dim=-1, keepdim=True) # (batch_size, sequence_length, 1)
        rms = torch.sqrt(x_sqrd_mean + self.eps) # (batch_size, sequence_length, 1)
        # (batch_size, sequence_length, d_model) * (d_model, ) / (batch_size, sequence_length, 1)
        result = x * self.gain / rms # (batch_size, sequence_length, d_model)
        return result.to(in_dtype)


In [114]:
rms_norm = RMSNorm(1024)
rms_norm.forward(torch.randn(1024))

tensor([-0.8464, -0.1628,  0.3576,  ..., -0.1147, -0.3050,  1.0986],
       grad_fn=<DivBackward0>)

In [126]:
import torch

class SwiGLU(torch.nn.Module):
    w_1: Linear # (d_ff, d_model)
    w_2: Linear # (d_model, d_ff)
    w_3: Linear # (d_ff, d_model)

    def __init__(self, d_model: int, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        d_ff = round(8 * d_model / 3 / 64) * 64
        self.w_1 = Linear(d_model, d_ff, device, dtype)
        self.w_2 = Linear(d_ff, d_model, device, dtype)
        self.w_3 = Linear(d_model, d_ff, device, dtype)

    # x (batch_size, sequence_length, d_model)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w_2(self._silu(self.w_1(x)) * self.w_3(x))

    def _silu(self, x: torch.Tensor) -> torch.Tensor:
        return x * torch.sigmoid(x)

In [127]:
d_model: int = 1024
swiglu = SwiGLU(d_model)
swiglu.forward(torch.randn(256, d_model))

tensor([[ 1.0154,  0.0667, -0.1968,  ...,  0.2918, -0.0655, -0.5058],
        [ 0.1068, -0.0103,  0.4649,  ..., -0.1191,  0.1005, -0.3194],
        [ 0.4846,  0.0939, -0.4680,  ..., -0.0723,  0.4256,  0.2847],
        ...,
        [ 0.1377, -0.0487,  0.1247,  ..., -0.3462,  0.5105,  0.0934],
        [-0.1571,  0.3470, -0.1651,  ...,  0.4462, -0.0574,  0.0732],
        [-0.6304, -0.1252,  0.3190,  ..., -0.0235,  0.5926, -0.4541]],
       grad_fn=<ViewBackward0>)

In [213]:
import torch
from einops import rearrange

class RotaryPositionalEmbedding(torch.nn.Module):

    def __init__(self, theta: float, d_k: int, max_seq_len: int, device: torch.device | None = None):
        super().__init__()
        k = torch.arange(0, d_k, 2, device=device)
        positions = torch.arange(0, max_seq_len, 1, device=device).reshape(max_seq_len, 1)
        angles = positions / theta ** (k / d_k)
        self.register_buffer('sin', torch.sin(angles), persistent=False)
        self.register_buffer('cos', torch.cos(angles), persistent=False)

    # x (batch, seq_len, d_k), token_positions (batch, seq_len)
    def forward(self, x: torch.Tensor, token_positions: torch.Tensor) -> torch.Tensor:
        x = rearrange(x, "... seq_len (half two) -> ... seq_len half two", two=2)
        a = x[..., 0] # (..., seq, half)
        b = x[..., 1] # (..., seq, half)
        cos = self.cos[token_positions]
        sin = self.sin[token_positions]
        a_out = a * cos - b * sin
        b_out = a * sin + b * cos
        x_out = torch.stack([a_out, b_out], dim=-1) # (..., seq_len, d_k/2 2)
        x_out = rearrange(x_out, "... seq_len half two -> ... seq_len (half two)")
        return x_out

In [214]:
d_k = 4
theta = 10000
max_seq_len = 2
x = torch.tensor([[1., 0., 1., 0.], [1., 0., 1., 0.]])
rope = RotaryPositionalEmbedding(theta, d_k, max_seq_len) # theta, d_k, max_seq_len
rope.cos.shape # seq_len, d_k/2
token_positions = torch.arange(max_seq_len)
rope.forward(x, token_positions)

tensor([[1.0000, 0.0000, 1.0000, 0.0000],
        [0.5403, 0.8415, 0.9999, 0.0100]])

In [226]:
import torch

def softmax(x: torch.Tensor, dim_i: int) -> torch.Tensor:
    x_max = x.max(dim=-1, keepdim=True).values
    x_norm = x.subtract(x_max)
    return x_norm.exp() / x_norm.exp().sum(dim=dim_i, keepdim=True)

In [227]:
x = torch.randn(2, 3)
print(x)
y = x.max(dim=-1, keepdim=True).values
print(y)
print(x.subtract(y))
print(softmax(x, -1))

tensor([[ 0.5392, -0.3073,  1.0038],
        [ 1.6859,  0.0698, -1.4772]])
tensor([[1.0038],
        [1.6859]])
tensor([[-0.4646, -1.3111,  0.0000],
        [ 0.0000, -1.6161, -3.1631]])
tensor([[0.3311, 0.1420, 0.5269],
        [0.8058, 0.1601, 0.0341]])


In [233]:
torch.allclose(torch.softmax(x, dim=-1), softmax(x, -1))

True

In [304]:
import torch 
from jaxtyping import Bool, Float

d_k = 4
d_v = 3
keys = 2
queries = 2

Q: Float[torch.Tensor, " ... queries d_k"] = torch.randn(keys, d_k)
K: Float[torch.Tensor, " ... keys d_k"] = torch.randn(queries, d_k)
V: Float[torch.Tensor, " ... keys d_v"] = torch.randn(keys, d_v)
mask: Bool[torch.Tensor, " ... queries keys"] | None = (torch.randn(queries, keys).uniform_() > 0.8)

In [308]:
import torch
from jaxtyping import Bool, Float
from einops import einsum
import math

def scaled_dot_product_attention(
        Q: Float[torch.Tensor, " ... queries d_k"],
        K: Float[torch.Tensor, " ... keys d_k"],
        V: Float[torch.Tensor, " ... keys d_v"],
        mask: Bool[torch.Tensor, " ... queries keys"] | None
) -> Float[torch.Tensor, " ... seq_len d_v"]:
    d_k = Q.shape[-1]
    qk = einsum(Q, K, "... queries d_k, ... keys d_k -> ... queries keys") / math.sqrt(d_k)
    if mask is not None:
        qk = qk.masked_fill(~mask, float("-Inf"))
    qk = softmax(qk, dim=-1)
    return einsum(qk, V, "... queries keys, ... keys d_v -> ... queries d_v")

In [309]:
attention(Q, K, V, mask)

tensor([[0.0000, 0.0000, 0.0000],
        [1.2504, 0.1861, 0.0448]])

In [310]:
torch.allclose(torch.nn.functional.scaled_dot_product_attention(Q, K, V), attention(Q, K, V))

True